# 1. Cleanup old processes

In [9]:
!pkill -u "$USER" -f '^clingcon '
!ps -aux | grep clingcon

unknown    36782  0.0  0.0  10484  3712 pts/1    Ss+  Mar20   0:00 bash -lc          echo "================ INSTANCE BEGIN ================" > logs/i_blue_circuit.lp-t_186400-cpu_64-20260320_155654.log;         echo "File: instances/blue_circuit.lp" >> logs/i_blue_circuit.lp-t_186400-cpu_64-20260320_155654.log;         echo "------------------------------------------------" >> logs/i_blue_circuit.lp-t_186400-cpu_64-20260320_155654.log;         cat instances/blue_circuit.lp >> logs/i_blue_circuit.lp-t_186400-cpu_64-20260320_155654.log;         echo >> logs/i_blue_circuit.lp-t_186400-cpu_64-20260320_155654.log;         echo "================ CLINGCON OUTPUT ================" >> logs/i_blue_circuit.lp-t_186400-cpu_64-20260320_155654.log;         systemd-run --user --scope -p MemoryMax=400G -p MemoryHigh=385G -p MemorySwapMax=0         clingcon --stats model2.lp game.lp instances/blue_circuit.lp --opt-mode=ignore -n 1 --quiet=1 -t 64 --time 186400 >> logs/i_blue_circuit.lp-t_186400-cpu_64-

# 2. Run clingcon on instance with timeout

In [15]:
import time, subprocess

timeMax = "1200"
instanceName = "green_circuit.lp"
coreCount = "64"

outfile = f"i_{instanceName}-t_{timeMax}-cpu_{coreCount}-{time.strftime('%Y%m%d_%H%M%S')}.log"
print(outfile)

sessionName = "asp"

inner_cmd = f"""
    bash -lc '
        echo "================ INSTANCE BEGIN ================" > logs/{outfile};
        echo "File: instances/{instanceName}" >> logs/{outfile};
        echo "------------------------------------------------" >> logs/{outfile};
        cat instances/{instanceName} >> logs/{outfile};
        echo >> logs/{outfile};
        echo "================ CLINGCON OUTPUT ================" >> logs/{outfile};
        systemd-run --user --scope -p MemoryMax=400G -p MemoryHigh=385G -p MemorySwapMax=0 \
        clingcon --stats model2.lp game.lp instances/{instanceName} --opt-mode=opt -n 0 --quiet=1 -t {coreCount} --time {timeMax} >> logs/{outfile} 2>&1
        '
"""

subprocess.run(["tmux", "new-session", "-d", "-s", sessionName, inner_cmd], check=True)
print(f"Started tmux session: {sessionName}")
print(f"Attach with: tmux attach -d -t {sessionName}")

i_green_circuit.lp-t_186400-cpu_64-20260322_182130.log
Started tmux session: asp
Attach with: tmux attach -d -t asp


# 3. Visualize

In [13]:
from clingcon_plotly_viewer import ClingconAssignmentViewer

#viewer = ClingconAssignmentViewer.from_log("logs/i_red_circuit.lp-t_86400-cpu_32-20260318_172649.log")
viewer = ClingconAssignmentViewer.from_log("logs/" + outfile)
ui = viewer.interactive_explorer_widget(cols=4, show_zeros_3d=True)
display(ui)